# Vehicle TXT → 1-Car Chunks (No Overlap) + Storage for Dense/Sparse/Hybrid Retrieval

This notebook:
- Reads a raw `.txt` file and splits it into **1 chunk = 1 vehicle** (no overlap)
- Builds a clean document schema: `doc_id`, `title`, `text`
- Writes chunks to disk in retrieval-friendly formats:
  - `chunks.jsonl` (recommended canonical format)
- Starter code for:
  - **Sparse retrieval (BM25)**
  - **Dense retrieval (FAISS + embeddings)**

> Assumption: vehicles are separated by **blank line blocks** (e.g., 2+ newlines).

In [12]:
# --- Imports
import re
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Tuple

In [13]:
# --- Config
RAW_TXT_PATH = "data\\raw\\vehicle_knowledge_base.txt"         
OUT_DIR = Path("data\\processed")   # output folder

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "chunks").mkdir(parents=True, exist_ok=True)

print("Output dir:", OUT_DIR.resolve())

Output dir: C:\Users\Roninasaurus\Documents\Mayur_Learning\Lambton\AIML_Capstone\rag_system_evaluation\data\processed


In [14]:
# --- IO utils (kept local; no dependency on project utils)
def read_text(path: str) -> str:
    return Path(path).read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")

def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def write_text(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")

In [15]:
# Title extraction: "The 2011 BMW 1 Series M is a ..."
TITLE_RE = re.compile(r"^(The\s+\d{4}\s+.+?)\s+is\s+a\s+", re.IGNORECASE)

def _make_doc_id(title: str, idx: int) -> str:
    safe = re.sub(r"[^a-zA-Z0-9]+", "_", title.strip()).strip("_").lower()
    return f"{safe}_{idx:05d}"

In [16]:
def parse_vehicle_docs(raw_txt_path: str) -> List[Dict[str, Any]]:
    raw = read_text(raw_txt_path)

    # Split on blank-line blocks.
    # This pattern splits when there are 2+ blank lines.
    blocks = [b.strip() for b in re.split(r"\n\s*\n\s*\n+", raw) if b.strip()]

    docs: List[Dict[str, Any]] = []
    for i, block in enumerate(blocks):
        first_line = block.splitlines()[0].strip()
        m = TITLE_RE.match(first_line)
        title = m.group(1).strip() if m else first_line[:80]

        doc_id = _make_doc_id(title, i)

        # Normalize whitespace inside each chunk (keeps chunk boundaries, improves retrieval)
        text = re.sub(r"[ \t]+", " ", block).strip()

        docs.append({"doc_id": doc_id, "title": title, "text": text})

    return docs

docs = parse_vehicle_docs(RAW_TXT_PATH)
print("Parsed vehicle docs:", len(docs))
print(docs[0] if docs else "No docs found")

Parsed vehicle docs: 11914
{'doc_id': 'the_2011_bmw_1_series_m_00000', 'title': 'The 2011 BMW 1 Series M', 'text': 'The 2011 BMW 1 Series M is a compact coupe.\nIt features a 6-cylinder premium unleaded engine producing 335 horsepower.\nThe transmission is manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: factory tuner, luxury, high-performance.\nFuel efficiency is rated at 19 city MPG and 26 highway MPG.\nIt has a popularity score of 3916.\nThe MSRP for this vehicle is $46,135.'}


In [ ]:
# Persist chunks 

# 1) Canonical JSONL (best for later pipelines)
jsonl_path = OUT_DIR / "chunks.jsonl"
write_jsonl(jsonl_path, docs)
print("Wrote:", jsonl_path)

# 2) one .txt per car (easy inspection / debugging)
for d in docs:
    write_text(OUT_DIR / "chunks" / f"{d['doc_id']}.txt", d["text"] + "\n")

print("Wrote per-doc txt files to:", (OUT_DIR / "chunks").resolve())

Wrote: data\processed\chunks.jsonl


## Sparse Retrieval: BM25 (self-contained)

Below is a small BM25 implementation (no external deps).  
It builds an index from `docs` and supports retrieval with scores.

In [18]:
def simple_tokenize(text: str) -> List[str]:
    # Lowercase, keep alphanumerics as tokens
    return re.findall(r"[a-z0-9]+", text.lower())

class BM25Index:
    def __init__(self, docs: List[Dict[str, Any]], k1: float = 1.5, b: float = 0.75):
        self.docs = docs
        self.k1 = k1
        self.b = b

        self.doc_tokens = [simple_tokenize(d["text"]) for d in docs]
        self.doc_lens = [len(toks) for toks in self.doc_tokens]
        self.avgdl = (sum(self.doc_lens) / max(1, len(self.doc_lens)))

        # Document frequencies
        df = {}
        for toks in self.doc_tokens:
            for term in set(toks):
                df[term] = df.get(term, 0) + 1
        self.df = df
        self.N = len(docs)

        # IDF with BM25+ style smoothing
        self.idf = {}
        for term, n_q in df.items():
            self.idf[term] = math.log(1 + (self.N - n_q + 0.5) / (n_q + 0.5))

        # Term frequencies per doc
        self.tfs = []
        for toks in self.doc_tokens:
            tf = {}
            for t in toks:
                tf[t] = tf.get(t, 0) + 1
            self.tfs.append(tf)

    def score(self, query: str, doc_idx: int) -> float:
        q_tokens = simple_tokenize(query)
        tf = self.tfs[doc_idx]
        dl = self.doc_lens[doc_idx]
        score = 0.0

        for term in q_tokens:
            if term not in tf:
                continue
            idf = self.idf.get(term, 0.0)
            f = tf[term]
            denom = f + self.k1 * (1 - self.b + self.b * (dl / (self.avgdl or 1.0)))
            score += idf * (f * (self.k1 + 1)) / (denom or 1.0)

        return score

    def search(self, query: str, k: int = 5) -> List[Tuple[Dict[str, Any], float]]:
        scored = [(self.docs[i], self.score(query, i)) for i in range(self.N)]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

bm25 = BM25Index(docs)
bm25.search("rear drive manual 335 horsepower", k=3)

[({'doc_id': 'the_2014_bmw_z4_11896',
   'title': 'The 2014 BMW Z4',
   'text': 'The 2014 BMW Z4 is a compact convertible.\nIt features a 6-cylinder premium unleaded engine producing 335 horsepower.\nThe transmission is automated_manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: luxury, high-performance.\nFuel efficiency is rated at 17 city MPG and 24 highway MPG.\nIt has a popularity score of 3916.\nThe MSRP for this vehicle is $65,800.'},
  7.340057241199167),
 ({'doc_id': 'the_2015_bmw_z4_11899',
   'title': 'The 2015 BMW Z4',
   'text': 'The 2015 BMW Z4 is a compact convertible.\nIt features a 6-cylinder premium unleaded engine producing 335 horsepower.\nThe transmission is automated_manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: luxury, high-performance.\nFuel efficiency is rated at 17 city MPG and 24 highway MPG.\nIt has a popularity score of 3916.\nThe MSRP for this vehicle is $65,800.'},
  7.34

## Dense Retrieval: Embeddings + FAISS (starter)

- an embedding model (e.g., `sentence-transformers`)
- FAISS (`faiss-cpu`)

In [19]:
import faiss
from sentence_transformers import SentenceTransformer

In [20]:
def build_faiss_index(docs: List[Dict[str, Any]], model_name: str = "all-MiniLM-L6-v2"):
    if faiss is None or SentenceTransformer is None:
        raise RuntimeError("Missing dependencies: faiss and/or sentence-transformers.")

    model = SentenceTransformer(model_name)
    texts = [d["text"] for d in docs]
    emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine similarity via inner product on normalized vectors
    index.add(emb)

    return model, index, emb

model, faiss_index, doc_embeddings = build_faiss_index(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 449.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 373/373 [02:16<00:00,  2.74it/s]


In [21]:
def faiss_search(query: str, model, index, docs: List[Dict[str, Any]], k: int = 5):
    q = model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(q, k)
    results = []
    for score, i in zip(scores[0].tolist(), idxs[0].tolist()):
        if i == -1:
            continue
        results.append((docs[i], float(score)))
    return results

faiss_search("luxury compact coupe 335 hp", model, faiss_index, docs, k=3)

[({'doc_id': 'the_1992_mercedes_benz_300_class_00180',
   'title': 'The 1992 Mercedes-Benz 300-Class',
   'text': 'The 1992 Mercedes-Benz 300-Class is a midsize coupe.\nIt features a 6-cylinder regular unleaded engine producing 217 horsepower.\nThe transmission is automatic with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: luxury.\nFuel efficiency is rated at 15 city MPG and 21 highway MPG.\nIt has a popularity score of 617.\nThe MSRP for this vehicle is $2,248.'},
  0.5663812160491943),
 ({'doc_id': 'the_2004_maserati_coupe_03030',
   'title': 'The 2004 Maserati Coupe',
   'text': 'The 2004 Maserati Coupe is a compact coupe.\nIt features a 8-cylinder premium unleaded engine producing 390 horsepower.\nThe transmission is manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: exotic, luxury, high-performance.\nFuel efficiency is rated at 10 city MPG and 15 highway MPG.\nIt has a popularity score of 238.\nThe MSRP for 